<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/llm/Create-Embeddings-out-of-an-LLM/embeddings-q2.ipynb)

This problem is not auto-graded: run the notebook's own checks, then compare with the solution notebook.


# Create Embeddings out of an LLM

### Problem Statement
Your mission, should you choose to accept it, is to extract **meaningful sentence-level embeddings** using a pre-trained **causal language model (SmolLM2-135M)** on Amazon Reviews.

You're working with a **generative language model**, but you’re not here to generate Shakespeare. Instead, you’ll tap into its **hidden states** to get semantic embeddings that capture the essence of a review — the good, the bad, and the brutally honest.

---

### Requirements

1. **Load and Tokenize Text**
   - Use the `McAuley-Lab/Amazon-Reviews-2023` dataset (All_Beauty reviews, `raw/review_categories/All_Beauty.jsonl`).
   - Load ~10 sample reviews for testing (stream them; the full file is ~330 MB).
   - Tokenize them using `"HuggingFaceTB/SmolLM2-135M"` tokenizer.

2. **Extract Embeddings**
   - Run the tokenized batch through the model with `output_hidden_states=True`.
   - Access the **last hidden layer** from `outputs.hidden_states[-1]`.

3. **Compute Sentence Embeddings** — pool the `(batch, seq_len, hidden_dim)` states into one vector per review:
   - **Mean pooling**: average the final-layer token states, **excluding padding tokens** (use the attention mask).
   - **Last-token pooling**: an encoder such as BERT has a `[CLS]` token whose state is trained to summarize the input. A causal LM has no such token, but because of the causal mask its **last real token** is the only one that has attended to the whole text, so it is the natural analog (LLM-based embedding models such as E5-Mistral pool it, after contrastive fine-tuning). Which index that is depends on the padding side.
   - Mean pooling over a causal model weights early tokens, which saw little context, as much as late ones; SGPT uses a position-weighted mean for that reason.

4. **Find the cosine similarity for a given keyword**
   - Embed the keyword the same way and compute its cosine similarity with each review embedding.
   - Raw hidden states of a generative LM that was never trained for embeddings are anisotropic: almost all cosine similarities come out high. Compare the *ranking* of the reviews, not the absolute values.

---

### Constraints

- ❌ Do **not** use sentence-transformers or pre-built embedding tools like `bert-as-service`.
- ❌ Do **not** generate text (no `.generate()`).
- ✅ Use only Hugging Face's `AutoModelForCausalLM` and `AutoTokenizer`.
- ✅ Exclude padding tokens when computing average embeddings.
- ✅ Ensure everything runs on `cuda` if available.

---

<details>
  <summary>💡 Hint</summary>

```python
# Run model with hidden states
outputs = model(**tokenized_inputs, output_hidden_states=True, return_dict=True)

# Get the last hidden layer (batch_size, seq_len, hidden_dim)
last_hidden = outputs.hidden_states[-1]

# Use the attention mask to avoid averaging over padding
attention_mask = tokenized_inputs['attention_mask']  # (batch_size, seq_len)

# Compute masked average: zero out padding tokens
masked_embeddings = last_hidden * attention_mask.unsqueeze(-1)  # broadcast mask
summed = masked_embeddings.sum(dim=1)  # sum across tokens
count = attention_mask.sum(dim=1, keepdim=True)  # count of non-padding tokens

# Final sentence-level embeddings
sentence_embeddings = summed / count  # (batch_size, hidden_dim)
```

</details>

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# Load ~10 reviews from Amazon-Reviews-2023 (All_Beauty). Streaming the raw JSONL file
# downloads only the first few rows of the ~330 MB file and needs no dataset script.
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw/review_categories/All_Beauty.jsonl",
    split="train",
    streaming=True,
)
reviews = [row["text"] for row in dataset.take(10)]  # list of 10 review strings

In [ ]:
for i, review in enumerate(reviews[:3]):
    print(f"#{i + 1}: {review[:120]}")

In [ ]:
# Load SmolLM2-135M model and tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M")

# SmolLM2's tokenizer has no padding token; batching needs one. Reusing EOS is safe
# because padded positions are masked out by the attention mask (and pooling below).
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()
print(type(model).__name__, f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

In [ ]:
def embed(texts):
    """Return (mean-pooled, last-token) sentence embeddings, each (batch, hidden_dim)."""
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**enc, output_hidden_states=True)
    last_hidden = outputs.hidden_states[-1].float()          # (batch, seq_len, hidden_dim)
    mask = enc["attention_mask"].unsqueeze(-1).float()      # 1 = real token, 0 = padding

    # Mean pooling over the real tokens only
    mean_emb = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

    # Last-token pooling: in a causal model only the last real token has attended to
    # the whole text, so it plays the role of BERT's [CLS]. With right padding its
    # index is (number of real tokens - 1).
    last_idx = enc["attention_mask"].sum(dim=1) - 1
    last_emb = last_hidden[torch.arange(last_hidden.size(0), device=device), last_idx]
    return mean_emb, last_emb


sentence_embeddings, last_token_embeddings = embed(reviews)
print("Sentence embeddings shape:", sentence_embeddings.shape)  # (10, hidden_dim)

# --- Cosine similarity for a given keyword ---
keyword = "quality"
keyword_embedding, _ = embed([keyword])                          # (1, hidden_dim)

cosine_similarities = F.cosine_similarity(sentence_embeddings, keyword_embedding)  # (10,)

# Raw LLM hidden states are anisotropic: every cosine similarity tends to be high,
# so compare the ranking, not the absolute values.
for i in cosine_similarities.argsort(descending=True).tolist():
    print(f"{cosine_similarities[i].item():.4f}  {reviews[i][:100]}")